<a href="https://colab.research.google.com/github/egaeuni/2026-1-DL-Final-Project/blob/main/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"현재 할당된 장치: {device}")

In [ ]:
# ---------------------------------------------------------
# 1. 데이터 로드
# ---------------------------------------------------------
drive.mount('/content/drive')

DIR_PATH = '/content/drive/MyDrive/DeepLearning_Data/Final_Project'
train_path = os.path.join(DIR_PATH, 'train.csv')
test_path = os.path.join(DIR_PATH, 'test.csv')

raw_train_df = pd.read_csv(train_path)
raw_test_df = pd.read_csv(test_path)

print(f"최초 로드 - Train: {raw_train_df.shape} | Test: {raw_test_df.shape}")

In [ ]:
# ---------------------------------------------------------
# 2. 파생 변수 생성 함수
# ---------------------------------------------------------
def create_initial_features(df):
    df = df.copy()

    df['Datetime'] = pd.to_datetime(df[['Year', 'Month', 'Day', 'Hour']])
    df = df.sort_values('Datetime').reset_index(drop=True)

    T, H, V = df['Temp'], df['Humi'], df['Wind']
    df['DI']         = 0.81*T + 0.01*H*(0.99*T - 14.3) + 46.3
    df['Wind_Chill'] = 13.12 + 0.6215*T - 11.37*(V**0.16) + 0.3965*T*(V**0.16)
    df['Temp_Range'] = df['HighTemp'] - df['LowTemp']

    df['DayOfWeek']   = df['Datetime'].dt.dayofweek
    df['Is_Weekend']  = (df['DayOfWeek'] >= 5).astype(int)
    df['Is_Monday']   = (df['DayOfWeek'] == 0).astype(int)
    df['Is_Friday']   = (df['DayOfWeek'] == 4).astype(int)

    df['Hour_sin']    = np.sin(2*np.pi*df['Hour']/24)
    df['Hour_cos']    = np.cos(2*np.pi*df['Hour']/24)
    df['Month_sin']   = np.sin(2*np.pi*df['Month']/12)
    df['Month_cos']   = np.cos(2*np.pi*df['Month']/12)
    df['Day_sin']     = np.sin(2*np.pi*df['Day']/31)
    df['Day_cos']     = np.cos(2*np.pi*df['Day']/31)
    df['Weekday_sin'] = np.sin(2*np.pi*df['DayOfWeek']/7)
    df['Weekday_cos'] = np.cos(2*np.pi*df['DayOfWeek']/7)

    def get_season(m):
        if m in [3,4,5]: return 'Spring'
        elif m in [6,7,8]: return 'Summer'
        elif m in [9,10,11]: return 'Autumn'
        else: return 'Winter'
    df['Season'] = df['Month'].apply(get_season)

    def get_kepco_load_type(month, hour):
        if month in [6,7,8]:
            if 13<=hour<=17: return 'Peak'
            elif (9<=hour<=12) or (18<=hour<=22): return 'Mid'
            else: return 'Light'
        elif month in [11,12,1,2]:
            if (9<=hour<=12) or (16<=hour<=17): return 'Peak'
            elif (13<=hour<=15) or (18<=hour<=22): return 'Mid'
            else: return 'Light'
        else:
            if (9<=hour<=12) or (13<=hour<=17) or (18<=hour<=22): return 'Mid'
            else: return 'Light'
    df['Load_Type'] = df.apply(
        lambda r: get_kepco_load_type(r['Month'], r['Hour']), axis=1
    )

    kr_holidays = holidays.KR(years=[2015,2016,2017,2018])
    df['Is_Holiday'] = df['Datetime'].apply(lambda x: 1 if x in kr_holidays else 0)

    unique_dates  = df['Datetime'].dt.date.unique()
    rest_day_dict = {
        d: 1 if d.weekday()>=5 or d in kr_holidays else 0
        for d in unique_dates
    }

    df['Temp_Date']      = df['Datetime'].dt.date
    df.drop('Temp_Date', axis=1, inplace=True)

    df = add_kma_warning_features(df)

    df['Is_Jangma'] = (
        (df['Month'].isin([6,7])) & (df['Humi'] >= 80)
    ).astype(int)

    lags = [1, 24, 72, 168]
    for lag in lags:
        df[f'Temp_Lag{lag}']  = df['Temp'].shift(lag)
        df[f'DI_Lag{lag}']    = df['DI'].shift(lag)
        df[f'Humi_Lag{lag}']  = df['Humi'].shift(lag)

    windows = [6, 12, 24]
    for w in windows:
        df[f'Temp_RollMean_{w}'] = df['Temp'].rolling(w).mean()
        df[f'Temp_RollMax_{w}']  = df['Temp'].rolling(w).max()
        df[f'Temp_RollMin_{w}']  = df['Temp'].rolling(w).min()

    df = df.bfill()
    return df

# ---------------------------------------------------------
# 3. 전처리 파이프라인 실행
# ---------------------------------------------------------
print("\n파생 변수 생성을 시작합니다...")

train_features_step1 = create_initial_features(raw_train_df)
test_features_step1 = create_initial_features(raw_test_df)

print(f"전처리 완료 - Train: {train_features_step1.shape} | Test: {test_features_step1.shape}")

In [ ]:
# ---------------------------------------------------------
#  타겟 변수(UsedPower)의 자기상관성(Autocorrelation) 분석 및 시각화
# ---------------------------------------------------------
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf

# 1. 주요 시차(Lag)별 자기상관 계수(ACF) 수치 계산
acf_values = acf(raw_train_df['UsedPower'], nlags=168)

print("=== 주요 시차(Lag)별 자기상관 계수(ACF) ===")
print(f"Lag 1   (1시간 전) : {acf_values[1]:.4f}")
print(f"Lag 24  (하루 전)   : {acf_values[24]:.4f}")
print(f"Lag 48  (이틀 전)   : {acf_values[48]:.4f}")
print(f"Lag 168 (일주일 전) : {acf_values[168]:.4f}")

# 2. ACF 및 PACF 그래프 시각화 (직관적인 확인을 위해 48시간 범위로 설정)
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# 자기상관함수(ACF) 그래프: 전체적인 주기성 확인
plot_acf(raw_train_df['UsedPower'], lags=48, ax=axes[0], title="Autocorrelation (ACF) - UsedPower")
axes[0].set_xlabel("Lags (Hours)")
axes[0].set_ylabel("ACF")
axes[0].grid(True, linestyle='--', alpha=0.5)

# 부분자기상관함수(PACF) 그래프: 다른 시차의 영향을 제거한 순수 상관관계 확인
plot_pacf(raw_train_df['UsedPower'], lags=48, ax=axes[1], title="Partial Autocorrelation (PACF) - UsedPower")
axes[1].set_xlabel("Lags (Hours)")
axes[1].set_ylabel("PACF")
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. 데이터 복사 및 Datetime 타입 변환
df_plot = train_features_step1.copy()
df_plot['Datetime'] = pd.to_datetime(df_plot['Datetime'])

plt.figure(figsize=(14, 6))
sns.set_style("whitegrid")

# 3. 산점도(Scatter Plot) 그리기
scatter = plt.scatter(
    x=df_plot['Datetime'],
    y=df_plot['Temp'],
    c=df_plot['Temp'],
    cmap='RdYlBu_r',
    alpha=0.5,
    s=10
)

# 4. 우측에 컬러바(온도계 역할) 추가
cbar = plt.colorbar(scatter)
cbar.set_label('Temperature (°C)', fontsize=12, fontweight='bold')

# 5. 핵심 단서: 시각적 강조선 추가 (한파와 폭염 기준선)
plt.axhline(0, color='blue', linestyle='--', linewidth=1.5, label='Freezing Point (0°C)')
plt.axhline(30, color='red', linestyle='--', linewidth=1.5, label='Heatwave Warning (30°C)')

plt.title('Temperature Distribution Over Time', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Date', fontsize=12, fontweight='bold')
plt.ylabel('Temperature (°C)', fontsize=12, fontweight='bold')
plt.legend(loc='upper left', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 1. 데이터 복사 및 날짜 처리
df_plot = train_features_step1.copy()
df_plot['Datetime'] = pd.to_datetime(df_plot['Datetime'])
df_plot['Month'] = df_plot['Datetime'].dt.month

# 7월과 8월을 '여름(장마)'로 구분하는 마스크 생성
is_summer = df_plot['Month'].isin([7, 8])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.set_style("whitegrid")

# ==========================================
# [왼쪽 그래프] 기온 vs 습도 산점도 (여름철 군집 강조)
# ==========================================
# 7~8월이 아닌 데이터
sns.scatterplot(
    x=df_plot[~is_summer]['Temp'],
    y=df_plot[~is_summer]['Humi'],
    color='lightgray', alpha=0.4, s=15, ax=axes[0], label='Other Months'
)
# 7~8월 장마철 데이터
sns.scatterplot(
    x=df_plot[is_summer]['Temp'],
    y=df_plot[is_summer]['Humi'],
    color='red', alpha=0.6, s=15, ax=axes[0], label='Jul & Aug (Jangma)'
)

axes[0].set_title('Temp vs Humidity: The Summer Anomaly', fontsize=15, fontweight='bold', pad=10)
axes[0].set_xlabel('Temperature (°C)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Humidity (%)', fontsize=12, fontweight='bold')
axes[0].legend(loc='upper left', fontsize=11)

# 고온다습 구역에 점선 박스 그려서 시선 집중시키기
axes[0].axvline(25, color='black', linestyle='--', alpha=0.5)
axes[0].axhline(70, color='black', linestyle='--', alpha=0.5)
axes[0].text(30, 95, 'High Temp & High Humi', color='darkred', fontsize=11, fontweight='bold', ha='center')


# ==========================================
# [오른쪽 그래프] 월별 기온 및 습도 평균 추이 (Dual Y-axis)
# ==========================================
monthly_avg = df_plot.groupby('Month')[['Temp', 'Humi']].mean()

ax1 = axes[1]
ax2 = ax1.twinx() # 습도를 위한 두 번째 Y축 생성

# 기온 꺾은선
line1 = ax1.plot(monthly_avg.index, monthly_avg['Temp'], color='red', marker='o', linewidth=2.5, label='Average Temp')
# 습도 꺾은선
line2 = ax2.plot(monthly_avg.index, monthly_avg['Humi'], color='blue', marker='s', linewidth=2.5, label='Average Humidity')

ax1.axvspan(6.5, 8.5, color='gray', alpha=0.15, label='Monsoon Season')

ax1.set_title('Monthly Average Trend', fontsize=15, fontweight='bold', pad=10)
ax1.set_xlabel('Month', fontsize=12, fontweight='bold')
ax1.set_ylabel('Temperature (°C)', color='red', fontsize=12, fontweight='bold')
ax2.set_ylabel('Humidity (%)', color='blue', fontsize=12, fontweight='bold')
ax1.set_xticks(range(1, 13))

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

def check_all_features_autocorrelation(df):
    # 1. 원본 데이터 복사 및 정렬
    df_sorted = df.sort_values('Datetime').reset_index(drop=True)

    # 2. 문자열 변수(Season, Load_Type)를 숫자로 원-핫 인코딩
    categorical_cols = [col for col in ['Season', 'Load_Type'] if col in df_sorted.columns]
    if categorical_cols:
        df_sorted = pd.get_dummies(df_sorted, columns=categorical_cols, dtype=int)

    # 3. 조사에서 제외할 컬럼 지정
    exclude_cols = ['id', 'Datetime', 'UsedPower']
    features_to_analyze = [col for col in df_sorted.columns if col not in exclude_cols]

    # 4. 확인할 과거 지연 시점
    target_lags = [1, 12, 24, 48, 72, 168]

    autocorr_results = {}

    # 5. 모든 변수를 돌면서 각 시점별 피어슨 자기상관 계수 계산
    for feature in features_to_analyze:
        lag_scores = {}
        for lag in target_lags:
            current_series = df_sorted[feature]
            lagged_series = df_sorted[feature].shift(lag)

            # 결측치를 제외하고 상관계수 계산
            corr = current_series.corr(lagged_series, method='pearson')
            lag_scores[f'T-{lag}'] = round(corr, 4) if not pd.isna(corr) else 0.0

        autocorr_results[feature] = lag_scores

    # 6. 결과를 보기 좋은 테이블로 반환
    autocorr_df = pd.DataFrame(autocorr_results).T

    # 분석하기 편하게 T-1(직전 상관성) 기준으로 내림차순 정렬
    autocorr_df = autocorr_df.sort_values(by='T-1', ascending=False)

    return autocorr_df

# ---------------------------------------------------------
# 실행 및 결과 출력
# ---------------------------------------------------------
all_features_autocorr = check_all_features_autocorrelation(train_features_step1)

print("=== [전수조사] 모든 파생 변수의 시점(Lag)별 자기상관성 요약 테이블 ===")
all_features_autocorr